In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA is available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version PyTorch was built with: {torch.version.cuda}")
    print(f"Current CUDA device name: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.3.0+cu121
CUDA is available: True
CUDA version PyTorch was built with: 12.1
Current CUDA device name: NVIDIA GeForce RTX 3060 Ti


In [ ]:
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from bisect import bisect_left
from dateutil.relativedelta import relativedelta
from sklearn.model_selection import KFold
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
import dgl.nn

# --- 데이터 전처리 함수 시작 ---
def calculate_age(born, trans_date):
    """생년월일과 거래 날짜를 이용하여 나이를 계산합니다."""
    return relativedelta(trans_date, born).years

def count_past_days_fast(df, days):
    """
    각 거래 시점을 기준으로 과거 'days' 기간 동안의 거래 횟수를 빠르게 계산합니다.
    df는 'cc_num'과 'trans_date_trans_time'으로 정렬되어 있어야 합니다.
    """
    results = []
    grouped = df.groupby('cc_num')

    for _, group in grouped:
        times = group['trans_date_trans_time'].tolist()
        counts = []
        for i, t in enumerate(times):
            start_time = t - pd.Timedelta(days=days)
            left_idx = bisect_left(times, start_time)
            right_idx = i
            counts.append(right_idx - left_idx)
        results.extend(counts)
    return results

def base_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    원본 데이터프레임에 기본적인 특징 엔지니어링을 적용합니다.
    로그 변환, 표준화, 시간 특징, 거래 빈도, 시간 간격, 나이 등을 포함합니다.
    그래프 구축에 필요한 ID 컬럼들은 유지합니다.
    """
    df_p = df.copy()

    # 불필요한 컬럼 제거 (그래프 구축에 사용되지 않는 원본 컬럼들)
    drop_cols_initial = ['Unnamed: 0', 'unix_time', 'trans_num', 'first', 'last', 'merchant',
                         'street', 'merch_lat', 'merch_long', 'city_pop', 'lat', 'long', 'zip']
    df_p = df_p.drop(columns=drop_cols_initial, errors='ignore')

    # 로그 변환 (amt)
    df_p['amt_log'] = np.log1p(df_p['amt'])

    # 표준화 (amt_log)
    scaler = StandardScaler()
    df_p['amt_log_std'] = scaler.fit_transform(df_p[['amt_log']])

    # datetime 변환 및 시간 관련 특징 추출
    df_p['datetime'] = pd.to_datetime(df_p['trans_date_trans_time'])
    df_p['hour'] = df_p['datetime'].dt.hour
    df_p['day_of_week'] = df_p['datetime'].dt.dayofweek # Monday=0, Sunday=6
    df_p['month'] = df_p['datetime'].dt.month # 1월=1, 12월=12

    # 시간 특징 (Sin/Cos 변환)
    df_p['trans_hour_sin'] = np.sin(2 * np.pi * df_p['hour'] / 24)
    df_p['trans_hour_cos'] = np.cos(2 * np.pi * df_p['hour'] / 24)
    df_p['is_night'] = df_p['hour'].isin([22, 23, 0, 1, 2, 3]).astype(int)

    # 고위험 시간대 특징
    high_risk_days = [2, 3, 4]  # 수, 목, 금
    high_risk_hours = list(range(0, 5)) + [21, 22, 23] # 0~4시, 21~23시
    df_p['is_high_risk_period'] = (
        df_p['day_of_week'].isin(high_risk_days) & df_p['hour'].isin(high_risk_hours)
    ).astype(int)

    # 거래 정렬 (cc_num별로 시간 순서)
    df_p['trans_date_trans_time'] = pd.to_datetime(df_p['trans_date_trans_time'])
    df_p = df_p.sort_values(['cc_num', 'trans_date_trans_time']).reset_index(drop=True)

    # 과거 N일간 거래 횟수
    df_p['cnt_1d'] = count_past_days_fast(df_p, 1)
    df_p['cnt_7d'] = count_past_days_fast(df_p, 7)
    df_p['cnt_30d'] = count_past_days_fast(df_p, 30)

    # 거래 시간 간격 특징
    df_p['next_trans_time'] = df_p.groupby('cc_num')['trans_date_trans_time'].shift(-1)
    df_p['time_since_last_trans'] = (df_p['trans_date_trans_time'] - df_p.groupby('cc_num')['trans_date_trans_time'].shift(1)).dt.total_seconds()
    df_p['time_until_next_trans'] = (df_p['next_trans_time'] - df_p['trans_date_trans_time']).dt.total_seconds()
    df_p[['time_since_last_trans', 'time_until_next_trans']] = df_p[['time_since_last_trans', 'time_until_next_trans']].fillna(0)

    # 나이 관련 특징
    df_p['dob'] = pd.to_datetime(df_p['dob'])
    df_p['age'] = df_p.apply(lambda x: calculate_age(x['dob'], x['trans_date_trans_time']), axis=1)
    df_p['age_group'] = (df_p['age'] // 10) * 10

    # 도시-주 결합 특징
    df_p['city_state'] = df_p['city'] + ', ' + df_p['state']

    # 범주형 변수 원-핫 인코딩
    df_p = pd.get_dummies(df_p, columns=['category'], prefix='category', drop_first=True)
    df_p = pd.get_dummies(df_p, columns=['gender'], prefix='gender', drop_first=True)

    # 최종 드롭 컬럼 (그래프 구축 후 드롭될 컬럼들 포함)
    # cc_num, job 등 그래프/타겟인코딩에 필요한 컬럼은 여기서 드롭하지 않음
    final_features_drop_cols = ['trans_date_trans_time', 'city', 'state', 'dob', 'next_trans_time', 'datetime', 'time_until_next_trans', 'amt_log', 'hour', 'amt']
    df_p = df_p.drop(columns=final_features_drop_cols, errors='ignore')

    return df_p

def preprocess_data(df: pd.DataFrame,
                    hour: str = 'sincos',
                    high_risk_period: bool = False,
                    age_group: bool = False
                    ) -> pd.DataFrame:
    """
    특정 전처리 옵션에 따라 컬럼을 선택하거나 제거합니다.
    Args:
        df: 입력 DataFrame.
        hour: 시간 전처리 방법 ('sincos' 또는 'is_night').
        high_risk_period: 고위험 시간대 컬럼 사용 여부.
        age_group: 연령대 그룹 컬럼 사용 여부.
    """
    df_p = df.copy()

    if hour == 'sincos':
        df_p = df_p.drop(columns=['is_night'], errors='ignore')
    elif hour == 'is_night':
        df_p = df_p.drop(columns=['trans_hour_sin','trans_hour_cos'], errors='ignore')

    if not high_risk_period:
        df_p = df_p.drop(columns=['is_high_risk_period'], errors='ignore')

    if age_group:
        df_p = df_p.drop(columns=['age'], errors='ignore')
    else:
        df_p = df_p.drop(columns=['age_group'], errors='ignore')

    return df_p


c:\Users\ogong\anaconda3\Lib\site-packages\torchdata\datapipes\__init__.py:18: UserWarning: 
################################################################################
WARNING!
The 'datapipes', 'dataloader2' modules are deprecated and will be removed in a
future torchdata release! Please see https://github.com/pytorch/data/issues/1196
to learn more and leave feedback.
################################################################################

  deprecation_warning()


In [ ]:
# --- K-Fold, 전체 타겟 인코딩 스무딩 함수 시작 ---
def m_estimate_smoothing(mean, global_mean, count, m):
    """M-estimate 스무딩을 적용하여 평균을 계산합니다."""
    return (mean * count + global_mean * m) / (count + m)

def kfold_target_encoding(df, target_col, cat_col, n_splits=5, m_param=10):
    """
    K-Fold 교차 검증 방식의 타겟 인코딩을 M-estimate 스무딩과 함께 적용하고,
    기존 cat_col 컬럼에 OOF 인코딩된 값을 덮어씁니다.

    Args:
        df: 입력 데이터프레임.
        target_col: 타겟 컬럼 이름 (예: 'is_fraud').
        cat_col: 인코딩하고 값을 덮어쓸 범주형 컬럼 이름.
        n_splits: K-Fold 분할 개수.
        m_param: M-estimate 스무딩 파라미터.
    Returns:
        인코딩된 컬럼이 덮어씌워진 DataFrame.
    """
    df_processed = df.copy() # 원본 DataFrame을 직접 수정하지 않기 위해 복사본 사용
    global_mean = df_processed[target_col].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    # OOF (Out-Of-Fold) 방식 인코딩 (과적합 방지)
    # 기존 cat_col에 덮어씌울 값을 임시로 저장할 컬럼 생성
    temp_encoded_values = np.empty(len(df_processed))
    temp_encoded_values[:] = np.nan # 초기화

    for train_idx, valid_idx in kf.split(df_processed):
        train_fold = df_processed.iloc[train_idx]
        agg = train_fold.groupby(cat_col)[target_col].agg(['mean', 'count']).reset_index()
        agg['smoothed'] = m_estimate_smoothing(agg['mean'], global_mean, agg['count'], m_param)
        mapping = dict(zip(agg[cat_col], agg['smoothed']))

        # 유효성 검사 폴드의 해당 cat_col 위치에 스무딩된 값을 매핑
        temp_encoded_values[valid_idx] = df_processed.loc[valid_idx, cat_col].map(mapping).fillna(global_mean)

    # 모든 폴드에 대한 처리가 끝난 후, 원본 cat_col에 OOF 인코딩된 값을 덮어씁니다.
    df_processed[cat_col] = temp_encoded_values

    return df_processed

def all_target_encoding(df, target_col, cat_col, m_param=10, encoding_map=None, global_mean=None):
    df_processed = df.copy()

    if encoding_map is not None and global_mean is not None:
        # 이미 학습된 맵과 전역 평균이 주어진 경우 (주로 테스트 데이터 인코딩)
        current_global_mean = global_mean
        current_encoding_map = encoding_map
    else:
        # 새로운 맵을 계산해야 하는 경우 (주로 훈련 데이터에서 맵 생성)
        current_global_mean = df_processed[target_col].mean()
        agg = df_processed.groupby(cat_col)[target_col].agg(['mean', 'count']).reset_index()
        agg['smoothed'] = m_estimate_smoothing(agg['mean'], current_global_mean, agg['count'], m_param)
        current_encoding_map = dict(zip(agg[cat_col], agg['smoothed']))

    # 인코딩 값 덮어쓰기: 맵에 없는 값은 전역 평균으로 대체
    df_processed[cat_col] = df_processed[cat_col].map(current_encoding_map).fillna(current_global_mean)

    if encoding_map is None:
        # 맵을 새로 계산한 경우에만 맵과 전역 평균을 반환
        return df_processed, current_encoding_map, current_global_mean
    else:
        # 주어진 맵을 사용한 경우 데이터프레임만 반환
        return df_processed

In [ ]:
# global_id_maps = {
#     'cc_num_to_idx': {},
#     'category_to_idx': {},
#     'city_state_to_idx': {}
# }

# # 각 노드 타입의 고유 ID 개수를 저장하여 나중에 특징 생성에 사용
# # 이 값들은 훈련 데이터 기준으로 한 번만 설정됩니다.
# global_num_unique_nodes = {
#     'credit_card': 0,
#     'original_category': 0,
#     'city_state_node': 0
# }


# def build_heterogeneous_graph(df: pd.DataFrame, is_train: bool = True, id_maps: dict = None):
#     """
#     전처리된 DataFrame으로부터 DGL 이종 그래프를 생성합니다.
#     노드: transaction, credit_card, original_category, city_state_node
#     엣지: uses_card, used_in, in_category, contains_trans, in_city_state, contains_trans

#     Args:
#         df (pd.DataFrame): 전처리된 데이터프레임.
#         is_train (bool): 훈련 그래프를 생성하는지, 테스트 그래프를 생성하는지 여부.
#                          True이면 새로운 ID 매핑을 생성하고 global_id_maps에 저장합니다.
#                          False이면 기존 id_maps를 사용합니다.
#         id_maps (dict, optional): is_train=False일 때 사용할 기존 ID 매핑 딕셔너리.
#                                  (global_id_maps와 동일한 구조)

#     Returns:
#         tuple: (dgl.DGLGraph, torch.Tensor) - 생성된 그래프와 타겟 레이블.
#     """
#     graph_data = {}

#     # 1. 노드 ID 매핑
#     num_transactions = len(df)

#     # df_temp는 이 함수 내에서만 사용되는 df의 복사본
#     df_temp = df.copy()

#     if is_train:
#         # 훈련 시에는 새로운 고유 ID 매핑을 생성하고 global_id_maps에 저장
#         unique_cc_nums = df_temp['cc_num'].unique()
#         id_maps['cc_num_to_idx'] = {cc_num: i for i, cc_num in enumerate(unique_cc_nums)}
#         global_num_unique_nodes['credit_card'] = len(unique_cc_nums)

#         # 'category_'로 시작하는 컬럼들을 통해 original_category_name을 추출
#         category_cols = [col for col in df_temp.columns if col.startswith('category_')]
#         df_temp['original_category_name'] = df_temp[category_cols].apply(lambda row: row.index[row.argmax()].replace('category_', '') if row.any() else 'Unknown', axis=1)
#         unique_categories = df_temp['original_category_name'].unique()
#         id_maps['category_to_idx'] = {cat: i for i, cat in enumerate(unique_categories)}
#         global_num_unique_nodes['original_category'] = len(unique_categories)

#         unique_city_states = df_temp['city_state'].unique()
#         id_maps['city_state_to_idx'] = {cs: i for i, cs in enumerate(unique_city_states)}
#         global_num_unique_nodes['city_state_node'] = len(unique_city_states)
#     else:
#         # 테스트 시에는 기존에 훈련 시 생성된 ID 매핑을 사용
#         if id_maps is None:
#             raise ValueError("테스트 그래프 생성 시에는 'id_maps'를 반드시 제공해야 합니다.")

#         # 'category_'로 시작하는 컬럼들을 통해 original_category_name을 추출
#         category_cols = [col for col in df_temp.columns if col.startswith('category_')]
#         df_temp['original_category_name'] = df_temp[category_cols].apply(lambda row: row.index[row.argmax()].replace('category_', '') if row.any() else 'Unknown', axis=1)


#     # num_nodes_dict는 훈련 시 결정된 전체 고유 노드 개수를 사용해야 함
#     num_nodes_dict = {
#         'transaction': num_transactions,
#         'credit_card': global_num_unique_nodes['credit_card'],
#         'original_category': global_num_unique_nodes['original_category'],
#         'city_state_node': global_num_unique_nodes['city_state_node']
#     }

#     # 2. 엣지 리스트 생성
#     # 매핑되지 않은 값은 -1로 채워서 DGL이 무시하도록 함
#     src_trans_uses_card = list(range(num_transactions))
#     dst_cc_uses_card = df_temp['cc_num'].map(id_maps['cc_num_to_idx']).fillna(-1).astype(int).tolist()
#     graph_data[('transaction', 'uses_card', 'credit_card')] = (src_trans_uses_card, dst_cc_uses_card)
#     graph_data[('credit_card', 'used_in', 'transaction')] = (dst_cc_uses_card, src_trans_uses_card)

#     src_trans_in_cat = list(range(num_transactions))
#     dst_cat_in_cat = df_temp['original_category_name'].map(id_maps['category_to_idx']).fillna(-1).astype(int).tolist()
#     graph_data[('transaction', 'in_category', 'original_category')] = (src_trans_in_cat, dst_cat_in_cat)
#     graph_data[('original_category', 'contains_trans_cat', 'transaction')] = (dst_cat_in_cat, src_trans_in_cat)

#     src_trans_in_cs = list(range(num_transactions))
#     dst_cs_in_cs = df_temp['city_state'].map(id_maps['city_state_to_idx']).fillna(-1).astype(int).tolist()
#     graph_data[('transaction', 'in_city_state', 'city_state_node')] = (src_trans_in_cs, dst_cs_in_cs)
#     graph_data[('city_state_node', 'contains_trans_cs', 'transaction')] = (dst_cs_in_cs, src_trans_in_cs)

#     # DGL 이종 그래프 생성
#     g = dgl.heterograph(graph_data, num_nodes_dict=num_nodes_dict)

#     # 3. 노드 특징 할당
#     # 'job'과 'city_state'가 이미 타겟 인코딩되어 수치형 특징으로 사용된다는 전제 하에 그대로 유지
#     transaction_features_cols = [
#         'amt_log_std', 'day_of_week', 'month', 'trans_hour_sin', 'trans_hour_cos',
#         'is_high_risk_period', 'cnt_1d', 'cnt_7d', 'cnt_30d', 'time_since_last_trans',
#         'age_group', 'age',
#         'category_food_dining', 'category_gas_transport', 'category_grocery_net',
#         'category_grocery_pos', 'category_health_fitness', 'category_home',
#         'category_kids_pets', 'category_misc_net', 'category_misc_pos',
#         'category_personal_care', 'category_shopping_net', 'category_shopping_pos',
#         'category_travel',
#         'gender_M',
#         'job', # 타겟 인코딩된 수치형 특징이므로 유지
#         'city_state', # 타겟 인코딩된 수치형 특징이므로 유지
#     ]

#     transaction_features_cols = [col for col in transaction_features_cols if col in df_temp.columns]

#     initial_transaction_features_df = df_temp[transaction_features_cols].copy()
#     for col in initial_transaction_features_df.columns:
#         # 이미 수치형이라 해도 pd.to_numeric은 안전함
#         initial_transaction_features_df[col] = pd.to_numeric(initial_transaction_features_df[col], errors='coerce')

#     initial_transaction_features = torch.tensor(
#         initial_transaction_features_df.fillna(0).to_numpy(dtype=np.float32),
#         dtype=torch.float32
#     )
#     g.nodes['transaction'].data['feat'] = initial_transaction_features

#     # 신용카드 노드 특징
#     cc_info_df = df_temp[['cc_num', 'job', 'city_state', 'gender_M']].drop_duplicates(subset=['cc_num']).set_index('cc_num')

#     # 훈련 시 생성된 unique_cc_nums의 순서에 맞춰 df를 정렬
#     # 이 과정에서 테스트 데이터에만 있는 cc_num은 NaN이 되며, 해당 row는 이후 fillna(0)으로 처리
#     # 이는 해당 cc_num의 특징이 모델에게 0으로 입력됨을 의미
#     cc_info_df = cc_info_df.reindex(id_maps['cc_num_to_idx'].keys())

#     credit_card_features_cols = ['job', 'city_state', 'gender_M'] # 타겟 인코딩된 수치형 특징이므로 유지
#     credit_card_features_cols = [col for col in credit_card_features_cols if col in cc_info_df.columns]

#     if not credit_card_features_cols:
#         # 특징 컬럼이 없는 경우, 훈련 데이터 기준 총 카드 개수에 맞춰 0으로 채워진 텐서 생성
#         initial_credit_card_features = torch.zeros(global_num_unique_nodes['credit_card'], 1, dtype=torch.float32)
#     else:
#         initial_credit_card_features_df = cc_info_df[credit_card_features_cols].copy()
#         for col in initial_credit_card_features_df.columns:
#             initial_credit_card_features_df[col] = pd.to_numeric(initial_credit_card_features_df[col], errors='coerce')

#         initial_credit_card_features = torch.tensor(
#             initial_credit_card_features_df.fillna(0).to_numpy(dtype=np.float32),
#             dtype=torch.float32
#         )
#     g.nodes['credit_card'].data['feat'] = initial_credit_card_features

#     # original_category 노드 특징 (훈련 데이터 기준 고유 개수로 원-핫 인코딩)
#     initial_category_features = torch.eye(global_num_unique_nodes['original_category'], dtype=torch.float32)
#     g.nodes['original_category'].data['feat'] = initial_category_features

#     # city_state_node 노드 특징 (훈련 데이터 기준 고유 개수로 원-핫 인코딩)
#     initial_city_state_features = torch.eye(global_num_unique_nodes['city_state_node'], dtype=torch.float32)
#     g.nodes['city_state_node'].data['feat'] = initial_city_state_features

#     target_labels = torch.tensor(df_temp['is_fraud'].values, dtype=torch.float32)

#     return g, target_labels

In [ ]:
global_id_maps = {
    'cc_num_to_idx': {},
    'category_to_idx': {},
    'city_state_to_idx': {}
}

# 각 노드 타입의 고유 ID 개수를 저장하여 나중에 특징 생성에 사용
# 이 값들은 훈련 데이터 기준으로 한 번만 설정됩니다.
global_num_unique_nodes = {
    'credit_card': 0,
    'original_category': 0,
    'city_state_node': 0
}

def build_heterogeneous_graph(df: pd.DataFrame, is_train: bool = True, id_maps: dict = None):
    """
    전처리된 DataFrame으로부터 DGL 이종 그래프를 생성합니다.
    노드: transaction, credit_card, original_category, city_state_node
    엣지: uses_card, used_in, in_category, contains_trans, in_city_state, contains_trans

    Args:
        df (pd.DataFrame): 전처리된 데이터프레임.
        is_train (bool): 훈련 그래프를 생성하는지, 테스트 그래프를 생성하는지 여부.
                         True이면 새로운 ID 매핑을 생성하고 global_id_maps에 저장합니다.
                         False이면 기존 id_maps를 사용합니다.
        id_maps (dict, optional): is_train=False일 때 사용할 기존 ID 매핑 딕셔너리.
                                 (global_id_maps와 동일한 구조)

    Returns:
        tuple: (dgl.DGLGraph, torch.Tensor) - 생성된 그래프와 타겟 레이블.
    """
    graph_data = {}

    # 1. 노드 ID 매핑
    num_transactions = len(df)

    # df_temp는 이 함수 내에서만 사용되는 df의 복사본
    df_temp = df.copy()

    if is_train:
        # 훈련 시에는 새로운 고유 ID 매핑을 생성하고 global_id_maps에 저장
        unique_cc_nums = df_temp['cc_num'].unique()
        id_maps['cc_num_to_idx'] = {cc_num: i for i, cc_num in enumerate(unique_cc_nums)}
        global_num_unique_nodes['credit_card'] = len(unique_cc_nums)

        # 'category_'로 시작하는 컬럼들을 통해 original_category_name을 추출
        category_cols = [col for col in df_temp.columns if col.startswith('category_')]
        df_temp['original_category_name'] = df_temp[category_cols].apply(lambda row: row.index[row.argmax()].replace('category_', '') if row.any() else 'Unknown', axis=1)
        unique_categories = df_temp['original_category_name'].unique()
        id_maps['category_to_idx'] = {cat: i for i, cat in enumerate(unique_categories)}
        global_num_unique_nodes['original_category'] = len(unique_categories)

        unique_city_states = df_temp['city_state'].unique()
        id_maps['city_state_to_idx'] = {cs: i for i, cs in enumerate(unique_city_states)}
        global_num_unique_nodes['city_state_node'] = len(unique_city_states)
    else:
        # 테스트 시에는 기존에 훈련 시 생성된 ID 매핑을 사용
        if id_maps is None:
            raise ValueError("테스트 그래프 생성 시에는 'id_maps'를 반드시 제공해야 합니다.")

        # 'category_'로 시작하는 컬럼들을 통해 original_category_name을 추출
        category_cols = [col for col in df_temp.columns if col.startswith('category_')]
        df_temp['original_category_name'] = df_temp[category_cols].apply(lambda row: row.index[row.argmax()].replace('category_', '') if row.any() else 'Unknown', axis=1)


    # num_nodes_dict는 훈련 시 결정된 전체 고유 노드 개수를 사용해야 함
    num_nodes_dict = {
        'transaction': num_transactions,
        'credit_card': global_num_unique_nodes['credit_card'],
        'original_category': global_num_unique_nodes['original_category'],
        'city_state_node': global_num_unique_nodes['city_state_node']
    }

    # 2. 엣지 리스트 생성
    # 매핑되지 않은 값은 -1로 채워서 DGL이 무시하도록 함
    src_trans_uses_card = list(range(num_transactions))
    dst_cc_uses_card = df_temp['cc_num'].map(id_maps['cc_num_to_idx']).fillna(-1).astype(int).tolist()
    graph_data[('transaction', 'uses_card', 'credit_card')] = (src_trans_uses_card, dst_cc_uses_card)
    graph_data[('credit_card', 'used_in', 'transaction')] = (dst_cc_uses_card, src_trans_uses_card)

    src_trans_in_cat = list(range(num_transactions))
    dst_cat_in_cat = df_temp['original_category_name'].map(id_maps['category_to_idx']).fillna(-1).astype(int).tolist()
    graph_data[('transaction', 'in_category', 'original_category')] = (src_trans_in_cat, dst_cat_in_cat)
    graph_data[('original_category', 'contains_trans_cat', 'transaction')] = (dst_cat_in_cat, src_trans_in_cat)

    src_trans_in_cs = list(range(num_transactions))
    dst_cs_in_cs = df_temp['city_state'].map(id_maps['city_state_to_idx']).fillna(-1).astype(int).tolist()
    graph_data[('transaction', 'in_city_state', 'city_state_node')] = (src_trans_in_cs, dst_cs_in_cs)
    graph_data[('city_state_node', 'contains_trans_cs', 'transaction')] = (dst_cs_in_cs, src_trans_in_cs)

    # DGL 이종 그래프 생성
    g = dgl.heterograph(graph_data, num_nodes_dict=num_nodes_dict)

    # 3. 노드 특징 할당
    # 'job'과 'city_state'가 이미 타겟 인코딩되어 수치형 특징으로 사용된다는 전제 하에 그대로 유지
    transaction_features_cols = [
        'amt_log_std', 'day_of_week', 'month', 'trans_hour_sin', 'trans_hour_cos',
        'is_high_risk_period', 'cnt_1d', 'cnt_7d', 'cnt_30d', 'time_since_last_trans',
        'age_group', 'age',
        'category_food_dining', 'category_gas_transport', 'category_grocery_net',
        'category_grocery_pos', 'category_health_fitness', 'category_home',
        'category_kids_pets', 'category_misc_net', 'category_misc_pos',
        'category_personal_care', 'category_shopping_net', 'category_shopping_pos',
        'category_travel',
        'gender_M',
        'job', # 타겟 인코딩된 수치형 특징이므로 유지
        'city_state', # 타겟 인코딩된 수치형 특징이므로 유지
    ]

    transaction_features_cols = [col for col in transaction_features_cols if col in df_temp.columns]

    initial_transaction_features_df = df_temp[transaction_features_cols].copy()
    for col in initial_transaction_features_df.columns:
        # 이미 수치형이라 해도 pd.to_numeric은 안전함
        initial_transaction_features_df[col] = pd.to_numeric(initial_transaction_features_df[col], errors='coerce')

    initial_transaction_features = torch.tensor(
        initial_transaction_features_df.fillna(0).to_numpy(dtype=np.float32),
        dtype=torch.float32
    )
    g.nodes['transaction'].data['feat'] = initial_transaction_features

    # 신용카드 노드 특징
    cc_info_df = df_temp[['cc_num', 'job', 'city_state', 'gender_M']].drop_duplicates(subset=['cc_num']).set_index('cc_num')

    # 훈련 시 생성된 unique_cc_nums의 순서에 맞춰 df를 정렬
    # 이 과정에서 테스트 데이터에만 있는 cc_num은 NaN이 되며, 해당 row는 이후 fillna(0)으로 처리
    # 이는 해당 cc_num의 특징이 모델에게 0으로 입력됨을 의미
    cc_info_df = cc_info_df.reindex(id_maps['cc_num_to_idx'].keys())

    credit_card_features_cols = ['job', 'city_state', 'gender_M'] # 타겟 인코딩된 수치형 특징이므로 유지
    credit_card_features_cols = [col for col in credit_card_features_cols if col in cc_info_df.columns]

    if not credit_card_features_cols:
        # 특징 컬럼이 없는 경우, 훈련 데이터 기준 총 카드 개수에 맞춰 0으로 채워진 텐서 생성
        initial_credit_card_features = torch.zeros(global_num_unique_nodes['credit_card'], 1, dtype=torch.float32)
    else:
        initial_credit_card_features_df = cc_info_df[credit_card_features_cols].copy()
        for col in initial_credit_card_features_df.columns:
            initial_credit_card_features_df[col] = pd.to_numeric(initial_credit_card_features_df[col], errors='coerce')

        initial_credit_card_features = torch.tensor(
            initial_credit_card_features_df.fillna(0).to_numpy(dtype=np.float32),
            dtype=torch.float32
        )
    g.nodes['credit_card'].data['feat'] = initial_credit_card_features

    # ORIGINAL_CATEGORY 노드 특징: ID 자체를 할당하여 nn.Embedding에서 사용할 수 있도록 합니다.
    # num_nodes_dict에 저장된 고유 카테고리 개수만큼 노드가 이미 DGL에 생성되어 있음.
    # 각 노드에 해당 인덱스를 그대로 특징으로 할당합니다.
    # 즉, 0번 카테고리 노드는 [0], 1번 카테고리 노드는 [1], ...
    initial_category_features = torch.arange(global_num_unique_nodes['original_category'], dtype=torch.long).unsqueeze(1)
    g.nodes['original_category'].data['feat'] = initial_category_features

    # CITY_STATE_NODE 노드 특징: ID 자체를 할당하여 nn.Embedding에서 사용할 수 있도록 합니다.
    # 위와 동일하게, num_nodes_dict에 저장된 고유 도시-주 개수만큼 노드가 이미 DGL에 생성되어 있음.
    # 각 노드에 해당 인덱스를 그대로 특징으로 할당합니다.
    initial_city_state_features = torch.arange(global_num_unique_nodes['city_state_node'], dtype=torch.long).unsqueeze(1)
    g.nodes['city_state_node'].data['feat'] = initial_city_state_features

    target_labels = torch.tensor(df_temp['is_fraud'].values, dtype=torch.float32)

    return g, target_labels

In [ ]:
# --- GNN-CL 모델 아키텍처 시작 ---
class GNNLayer(nn.Module):
    """
    GNN-CL 모델의 GNN 레이어입니다.
    DGL 그래프 객체와 모든 노드 타입의 차원 조정된 특징 딕셔너리를 입력받아 메시지 전달을 수행합니다.
    """
    def __init__(self, common_node_features_dim, gnn_out_features):
        super(GNNLayer, self).__init__()
        self.common_node_features_dim = common_node_features_dim
        self.gnn_out_features = gnn_out_features

        # Purification Network: 노드 특징을 정화하여 전달할 메시지를 제어
        # 입력 차원은 공통 노드 특징 차원
        self.mlp_purifier = nn.Sequential(
            nn.Linear(common_node_features_dim, common_node_features_dim),
            nn.ReLU(),
            nn.Linear(common_node_features_dim, common_node_features_dim),
            nn.Sigmoid() # 0-1 사이의 가중치를 생성하여 메시지를 필터링
        )

        # 각 관계 타입별 GraphConv 레이어
        # 모든 GraphConv 레이어의 in_features를 공통 노드 특징 차원으로 설정합니다.
        self.conv_layers = nn.ModuleDict({
            'uses_card': dgl.nn.GraphConv(common_node_features_dim, gnn_out_features),
            'used_in': dgl.nn.GraphConv(common_node_features_dim, gnn_out_features),
            'in_category': dgl.nn.GraphConv(common_node_features_dim, gnn_out_features),
            'contains_trans_cat': dgl.nn.GraphConv(common_node_features_dim, gnn_out_features),
            'in_city_state': dgl.nn.GraphConv(common_node_features_dim, gnn_out_features),
            'contains_trans_cs': dgl.nn.GraphConv(common_node_features_dim, gnn_out_features),
        })

        # Relationship Summarizer:
        # transaction 노드로 들어오는 엣지 타입의 수 계산
        # 현재는 'used_in', 'contains_trans_cat', 'contains_trans_cs' 세 가지
        num_incoming_etypes_to_trans = 3
        # 최종 summarizer의 입력 차원: 원본(정화된) transaction 특징 + 각 관계별 집계 특징 합
        self.final_summarizer_linear = nn.Linear(
            common_node_features_dim + gnn_out_features * num_incoming_etypes_to_trans,
            gnn_out_features
        )

    def forward(self, g, h_dict): # h_dict는 이미 차원 조정된 모든 노드 특징 딕셔너리
        with g.local_scope():
            # Purification Network 적용: 'transaction' 노드의 특징에만 적용
            transaction_features = h_dict['transaction']
            purification_weights = self.mlp_purifier(transaction_features)

            aggregated_h_per_relation = []

            # 각 관계 타입별로 메시지 전달 및 집계
            for etype, conv_layer in self.conv_layers.items():
                src_type, _, dst_type = g.canonical_etypes[g.get_etype_id(etype)]
                sg = g[src_type, etype, dst_type] # 해당 엣지 타입의 서브그래프

                # 소스 노드의 특징을 h_dict에서 가져옵니다.
                h_src = h_dict[src_type]

                # 'transaction' 노드에서 출발하는 메시지에만 purification 가중치를 적용합니다.
                if src_type == 'transaction':
                    h_src_purified = h_src * purification_weights
                else:
                    h_src_purified = h_src # 다른 노드들은 Purification 적용 안함

                # 목적지 노드가 'transaction'인 경우에만 집계를 수행합니다.
                if dst_type == 'transaction':
                    h_rel = conv_layer(sg, h_src_purified)
                    aggregated_h_per_relation.append(h_rel)
                else:
                    # 목적지 노드가 transaction이 아닌 엣지는 메시지를 전달만 하고
                    # GNNLayer의 최종 출력인 transaction_final_h에는 직접적으로 기여하지 않음.
                    # 하지만 conv_layer는 여전히 메시지 전달 학습에 참여.
                    # (여기서 반환값으로 transaction 노드 특징만 필요하다는 가정을 따름)
                    pass


            # 원본(정화된) transaction 특징과 'transaction' 노드로 집계된 특징들을 결합합니다.
            if not aggregated_h_per_relation:
                # incoming edge가 없는 경우, transaction_features만 사용 (하지만 이 모델에서는 최소 3개 있음)
                concatenated_h = transaction_features
            else:
                concatenated_h = torch.cat([transaction_features] + aggregated_h_per_relation, dim=1)

            # 최종 선형 레이어와 활성화 함수를 적용하여 현재 레이어의 임베딩을 계산합니다.
            h_current_transaction = F.relu(self.final_summarizer_linear(concatenated_h))

            # GNNLayer의 출력은 transaction 노드의 업데이트된 특징
            return h_current_transaction

In [ ]:
class CNNLayer(nn.Module):
    """
    GNN-CL 모델의 CNN 레이어입니다.
    각 노드의 GNN 출력을 시퀀스로 보고 특징을 추출합니다.
    """
    def __init__(self, in_features_from_gnn, out_channels, kernel_size):
        super(CNNLayer, self).__init__()
        self.conv1d = nn.Conv1d(in_channels=1, out_channels=out_channels,
                                 kernel_size=kernel_size, padding=kernel_size//2)
        # MaxPool1d는 사용하지 않고 AdaptiveMaxPool1d를 사용합니다.
        # self.pool = nn.MaxPool1d(kernel_size) # 제거

        self.adaptive_pool = nn.AdaptiveMaxPool1d(1) # 각 (C_out, L_pool)을 (C_out, 1)로 변환
        self.linear_transform = nn.Linear(out_channels, out_channels) # CNN의 최종 출력을 정규화

    def forward(self, x): # x.shape: (num_nodes, in_features_from_gnn)
        # Conv1d 입력 형태로 변환: (num_nodes, 1, in_features_from_gnn)
        x = x.unsqueeze(1) # (num_nodes, 1, features)
        x = F.relu(self.conv1d(x)) # (num_nodes, out_channels, L_conv)
        x = self.adaptive_pool(x) # (num_nodes, out_channels, 1) -> Global Max Pooling
        x = x.squeeze(2) # (num_nodes, out_channels)
        x = self.linear_transform(x) # (num_nodes, out_channels)
        return x

In [ ]:
class LSTMLayer(nn.Module):
    """
    GNN-CL 모델의 LSTM 레이어입니다.
    CNN 출력을 시퀀스로 보고 최종 예측을 수행합니다.
    노드 레벨 분류를 위해 (num_nodes, 1) 출력을 반환하도록 수정되었습니다.
    """
    def __init__(self, input_size, hidden_size, num_layers=1, bidirectional=True): # 논문 따라 bidirectional=True
        super(LSTMLayer, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=bidirectional)

        output_linear_input_size = hidden_size * 2 if bidirectional else hidden_size
        self.output_linear = nn.Linear(output_linear_input_size, 1) # 노드 레벨 예측
        # output_linear_input_size는 hidden_size * 2 임

    def forward(self, x): # x.shape: (num_nodes, input_size) (CNNLayer의 출력)
        # LSTM은 (batch_size, seq_len, input_size)를 기대
        # num_nodes를 batch_size로 보고, seq_len을 1로 가정
        x = x.unsqueeze(1) # x.shape: (num_nodes, 1, input_size)

        output, (h_n, c_n) = self.lstm(x) # output.shape: (num_nodes, 1, hidden_size * num_directions)

        # LSTM 출력에서 seq_len 차원 제거 후, 최종 분류 레이어 적용
        output_combined = self.output_linear(output.squeeze(1)) # output_combined.shape: (num_nodes, 1)

        return output_combined

In [ ]:
class GNN_CL_Model(nn.Module):
    """
    GNN-CL(Graph Neural Network for Contrastive Learning) 모델의 전체 아키텍처입니다.
    GNN, CNN, LSTM 구성 요소를 통합합니다.
    """
    def __init__(self,
                 node_info: dict, # 각 노드 타입의 원본 특징 차원 정보 { 'node_type': raw_dim }
                 common_node_feature_dim: int, # GNN 레이어가 기대하는 공통 특징 차원
                 gnn_out_features: int,
                 cnn_out_channels: int,
                 cnn_kernel_size: int,
                 lstm_hidden_size: int):
        super(GNN_CL_Model, self).__init__()

        self.common_node_feature_dim = common_node_feature_dim

        # 노드 타입별 특징을 GNN의 입력 차원(common_node_feature_dim)으로 투영하는 레이어
        self.node_feature_projectors = nn.ModuleDict()
        for ntype, raw_dim in node_info.items():
            if ntype in ['original_category', 'city_state_node']:
                self.node_feature_projectors[ntype] = nn.Embedding(raw_dim, common_node_feature_dim)
            else:
                self.node_feature_projectors[ntype] = nn.Linear(raw_dim, common_node_feature_dim)

        # GNN 레이어
        self.gnn_layer = GNNLayer(common_node_feature_dim, gnn_out_features)

        # CNN 레이어
        # CNN 입력 차원은 GNN 출력 차원 (transaction 노드의 최종 임베딩 차원)
        self.cnn_layer = CNNLayer(gnn_out_features, cnn_out_channels, cnn_kernel_size)

        # LSTM 레이어
        # LSTM 입력 차원은 GNN 출력 차원 (transaction 노드의 최종 임베딩 차원)
        self.lstm_layer = LSTMLayer(gnn_out_features, lstm_hidden_size)

        # 최종 분류기 (GNN, CNN, LSTM의 출력을 모두 사용하여 이진 분류)
        # GNN 출력: gnn_out_features (예: 64)
        # CNN 출력: cnn_out_channels (예: 32)
        # LSTM 출력: 1
        # 따라서 최종 입력 차원은 gnn_out_features + cnn_out_channels + 1
        final_classifier_input_dim = gnn_out_features + cnn_out_channels + 1
        self.final_classifier = nn.Linear(final_classifier_input_dim, 1)


    def forward(self, g):
        h_dict = {}
        for ntype, feat in g.ndata['feat'].items():
            if ntype in ['original_category', 'city_state_node']:
                h_dict[ntype] = self.node_feature_projectors[ntype](feat.squeeze(1))
            else:
                h_dict[ntype] = self.node_feature_projectors[ntype](feat)

        gnn_output_transaction = self.gnn_layer(g, h_dict)

        cnn_output = self.cnn_layer(gnn_output_transaction)

        lstm_output = self.lstm_layer(gnn_output_transaction)

        combined_output = torch.cat([gnn_output_transaction, cnn_output, lstm_output], dim=1)
        final_prediction_logits = self.final_classifier(combined_output)

        return final_prediction_logits

In [ ]:
# --- 1. 데이터 로드: fraudTrain.csv (학습 데이터) 및 fraudTest.csv (테스트 데이터) ---
train_df = pd.read_csv('./data/fraudTrain.csv')
test_df = pd.read_csv('./data/fraudTest.csv')

In [ ]:
# --- 2. base_df 전처리 실행 ---
print("--- 2. base_df 전처리 실행 (CTGAN 학습 이전에 원본 데이터에 적용) ---")
processed_train_df = base_df(train_df)
processed_test_df = base_df(test_df)
print(f"원본 raw_df에 base_df 적용 후 형태: {processed_train_df.shape}")
print(f"원본 test_df에 base_df 적용 후 형태: {processed_test_df.shape}")

--- 2. base_df 전처리 실행 (CTGAN 학습 이전에 원본 데이터에 적용) ---
원본 raw_df에 base_df 적용 후 형태: (1296675, 31)
원본 test_df에 base_df 적용 후 형태: (555719, 31)


In [ ]:
# --- 3. K-Fold 타겟 인코딩 (스무딩 적용) ---
categorical_cols_for_smoothing = ['city_state', 'job']
m_param_smoothing = 100

df_with_smoothed_features_train = processed_train_df.copy()
df_with_smoothed_features_test = processed_test_df.copy()

print("--- 훈련 데이터셋 K-Fold OOF 타겟 인코딩 시작 ---")
print(f"변환 전 'city_state' 컬럼의 데이터 타입 (훈련): {df_with_smoothed_features_train['city_state'].dtype}")
print(f"변환 전 'job' 컬럼의 데이터 타입 (훈련): {df_with_smoothed_features_train['job'].dtype}")

for cat_col in categorical_cols_for_smoothing:
    if cat_col in df_with_smoothed_features_train.columns:
        df_with_smoothed_features_train = kfold_target_encoding(
            df_with_smoothed_features_train, 'is_fraud', cat_col, m_param=m_param_smoothing
        )
        print(f"  - 훈련 데이터셋: '{cat_col}' 컬럼에 스무딩된 OOF 인코딩 값 덮어쓰기 완료.")
    else:
        print(f"  - 경고: 인코딩을 위한 컬럼 '{cat_col}'이 훈련 DataFrame에 없습니다. 건너뜀.")

print(f"K-Fold OOF 타겟 인코딩 후 훈련 DataFrame: {df_with_smoothed_features_train.shape}")
print(f"변환 후 'city_state' 컬럼의 데이터 타입 (훈련): {df_with_smoothed_features_train['city_state'].dtype}")
print(f"변환 후 'job' 컬럼의 데이터 타입 (훈련): {df_with_smoothed_features_train['job'].dtype}")
print("-" * 50)

print("\n--- 테스트 데이터셋 전체 타겟 인코딩 시작 (훈련 데이터 맵 사용) ---")
print(f"변환 전 'city_state' 컬럼의 데이터 타입 (테스트): {df_with_smoothed_features_test['city_state'].dtype}")
print(f"변환 전 'job' 컬럼의 데이터 타입 (테스트): {df_with_smoothed_features_test['job'].dtype}")

# 각 범주형 컬럼에 대해 훈련 데이터셋에서 인코딩 맵을 학습하고, 테스트 데이터셋에 적용
# 훈련 데이터셋으로부터 맵을 학습하는 과정
learned_encoding_maps = {}
learned_global_means = {}

for cat_col in categorical_cols_for_smoothing:
    if cat_col in processed_train_df.columns:
        # 훈련 데이터셋을 사용하여 맵과 글로벌 평균을 학습합니다.
        # 이 때 반환되는 df_processed는 사용하지 않고 맵과 평균만 저장합니다.
        _, learned_map, learned_global_mean = all_target_encoding(
            processed_train_df, 'is_fraud', cat_col, m_param=m_param_smoothing
        )
        learned_encoding_maps[cat_col] = learned_map
        learned_global_means[cat_col] = learned_global_mean
        print(f"  - 훈련 데이터셋으로부터 '{cat_col}'에 대한 인코딩 맵 학습 완료.")
    else:
        print(f"  - 경고: 훈련 데이터셋에 '{cat_col}' 컬럼이 없습니다. 테스트셋 인코딩을 건너뜀.")
        continue # 이 컬럼에 대한 테스트셋 인코딩 건너뛰기

    # 학습된 맵과 글로벌 평균을 사용하여 테스트 데이터셋을 인코딩합니다.
    if cat_col in df_with_smoothed_features_test.columns:
        df_with_smoothed_features_test = all_target_encoding(
            df_with_smoothed_features_test, 'is_fraud', cat_col=cat_col, m_param=m_param_smoothing,
            encoding_map=learned_encoding_maps[cat_col],
            global_mean=learned_global_means[cat_col]
        )
        print(f"  - 테스트 데이터셋: '{cat_col}' 컬럼에 스무딩된 값 덮어쓰기 완료 (훈련 맵 사용).")
    else:
        print(f"  - 경고: 테스트 데이터셋에 '{cat_col}' 컬럼이 없습니다. 건너뜜.")

print(f"K-Fold 타겟 인코딩 후 테스트 DataFrame: {df_with_smoothed_features_test.shape}")
print(f"변환 후 'city_state' 컬럼의 데이터 타입 (테스트): {df_with_smoothed_features_test['city_state'].dtype}")
print(f"변환 후 'job' 컬럼의 데이터 타입 (테스트): {df_with_smoothed_features_test['job'].dtype}")
print("-" * 50)

--- 훈련 데이터셋 K-Fold OOF 타겟 인코딩 시작 ---
변환 전 'city_state' 컬럼의 데이터 타입 (훈련): object
변환 전 'job' 컬럼의 데이터 타입 (훈련): object
  - 훈련 데이터셋: 'city_state' 컬럼에 스무딩된 OOF 인코딩 값 덮어쓰기 완료.
  - 훈련 데이터셋: 'job' 컬럼에 스무딩된 OOF 인코딩 값 덮어쓰기 완료.
K-Fold OOF 타겟 인코딩 후 훈련 DataFrame: (1296675, 31)
변환 후 'city_state' 컬럼의 데이터 타입 (훈련): float64
변환 후 'job' 컬럼의 데이터 타입 (훈련): float64
--------------------------------------------------

--- 테스트 데이터셋 전체 타겟 인코딩 시작 (훈련 데이터 맵 사용) ---
변환 전 'city_state' 컬럼의 데이터 타입 (테스트): object
변환 전 'job' 컬럼의 데이터 타입 (테스트): object
  - 훈련 데이터셋으로부터 'city_state'에 대한 인코딩 맵 학습 완료.
  - 테스트 데이터셋: 'city_state' 컬럼에 스무딩된 값 덮어쓰기 완료 (훈련 맵 사용).
  - 훈련 데이터셋으로부터 'job'에 대한 인코딩 맵 학습 완료.
  - 테스트 데이터셋: 'job' 컬럼에 스무딩된 값 덮어쓰기 완료 (훈련 맵 사용).
K-Fold 타겟 인코딩 후 테스트 DataFrame: (555719, 31)
변환 후 'city_state' 컬럼의 데이터 타입 (테스트): float64
변환 후 'job' 컬럼의 데이터 타입 (테스트): float64
--------------------------------------------------


In [ ]:
# --- 4. CTGAN을 통한 데이터 증강 ---
print("--- 4. CTGAN 데이터 증강 ---")

cc_num_avg_amt_map = df_with_smoothed_features_train.groupby('cc_num')['amt_log_std'].mean()
df_with_smoothed_features_train['cc_num_avg_amt'] = df_with_smoothed_features_train['cc_num'].map(cc_num_avg_amt_map)

df_fraud_only = df_with_smoothed_features_train[df_with_smoothed_features_train['is_fraud'] == 1].drop(columns=['cc_num'], errors='ignore')

augmented_data_size = 55000
epochs_ctgan = 300

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_fraud_only)

metadata.update_column(column_name='job', sdtype='numerical')
metadata.update_column(column_name='city_state', sdtype='numerical')
metadata.update_column(column_name='is_fraud', sdtype='categorical')
metadata.update_column(column_name='cc_num_avg_amt', sdtype='numerical')

print(f"CTGAN 초기화 중... {epochs_ctgan} 에포크.")
synthesizer = CTGANSynthesizer(metadata, enforce_rounding=False, enforce_min_max_values=False, epochs=epochs_ctgan)

print("CTGAN 모델을 df_for_ctgan_training에 학습시키는 중입니다...")
synthesizer.fit(df_fraud_only)
print("CTGAN 모델 학습 완료. 합성 데이터 생성 중...")

synthetic_data_without_cc_num = synthesizer.sample(num_rows=augmented_data_size)
synthetic_data_without_cc_num['is_fraud'] = 1

print(f"합성 데이터 {len(synthetic_data_without_cc_num)} 샘플 생성 완료 (cc_num 제외).")
print("합성 데이터프레임 (cc_num 제외, 파생 변수 포함) 상위 5행:")
print(synthetic_data_without_cc_num.head())
print("\n" + "="*50 + "\n")

--- 4. CTGAN 데이터 증강 ---
CTGAN 초기화 중... 300 에포크.
CTGAN 모델을 df_for_ctgan_training에 학습시키는 중입니다...


c:\Users\ogong\anaconda3\Lib\site-packages\sdv\single_table\base.py:163: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
c:\Users\ogong\anaconda3\Lib\site-packages\sdv\single_table\base.py:129: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


CTGAN 모델 학습 완료. 합성 데이터 생성 중...
합성 데이터 55000 샘플 생성 완료 (cc_num 제외).
합성 데이터프레임 (cc_num 제외, 파생 변수 포함) 상위 5행:
        job  is_fraud  amt_log_std  day_of_week  month  trans_hour_sin  \
0  0.007741         1     1.113314            0     12       -0.520385   
1  0.003145         1    -0.431764            2     12        0.463691   
2  0.005225         1     0.448280            6     12        0.748570   
3  0.009684         1    -0.500474            1      8       -0.264405   
4  0.009086         1     2.061212            5      6        0.003860   

   trans_hour_cos  is_night  is_high_risk_period  cnt_1d  ...  category_home  \
0        0.872139         1                    0       4  ...          False   
1        0.866074         1                    1       7  ...           True   
2        0.276597         1                    0       3  ...          False   
3        0.868650         1                    1       3  ...          False   
4        0.956755         1                    1  

In [ ]:
synthetic_data_without_cc_num.shape

(55000, 31)

In [ ]:
synthetic_data_without_cc_num['is_fraud'].unique()

array([1], dtype=int64)

In [ ]:
# --- 4-2 합성 데이터에 cc_num 재할당 ---

from sklearn.neighbors import NearestNeighbors

print("합성 데이터에 'cc_num' 재할당 중 (유사성 기반 매칭)...")

# cc_num_avg_amt, job, city_state, gender_M을 매칭 특성으로 사용
features_for_nn = ['cc_num_avg_amt', 'job', 'city_state', 'gender_M']

# 원본 데이터의 cc_num과 그에 해당하는 특성 매핑 준비
original_fraud_cc_nums_info = df_with_smoothed_features_train[df_with_smoothed_features_train['is_fraud'] == 1][['cc_num'] + features_for_nn].drop_duplicates()

# 특성 스케일링: 서로 다른 범위의 특성들이 유사성 계산에 동등하게 기여하도록 합니다.
scaler = StandardScaler()

# 원본 사기 거래 특성으로 스케일러 학습 및 변환
scaled_original_features = scaler.fit_transform(original_fraud_cc_nums_info[features_for_nn])

# NearestNeighbors 모델 학습
nn_model = NearestNeighbors(n_neighbors=1, algorithm='brute')
nn_model.fit(scaled_original_features)

# 합성 데이터의 해당 특성들을 스케일링
scaled_synthetic_features = scaler.transform(synthetic_data_without_cc_num[features_for_nn])

# 합성 데이터의 스케일링된 특성 값을 사용하여 원본 사기 cc_num에 매핑
distances, indices = nn_model.kneighbors(scaled_synthetic_features)

# 매핑된 실제 cc_num 값들을 가져옵니다.
mapped_cc_nums_from_fraud_pool = original_fraud_cc_nums_info.iloc[indices.flatten()]['cc_num'].values

# Series로 변환하여 synthetic_data_without_cc_num과 병합 준비
synthetic_cc_num_series_for_fraud = pd.Series(mapped_cc_nums_from_fraud_pool, name='cc_num', index=synthetic_data_without_cc_num.index)

# 합성된 사기 피처 데이터와 재할당된 cc_num을 결합하여 최종 합성 사기 데이터 완성
synthetic_fraud_data_final = pd.concat([synthetic_data_without_cc_num, synthetic_cc_num_series_for_fraud], axis=1)

# 6. 원본 데이터프레임 (정상 및 사기 거래 모두 포함)과 합성 사기 데이터프레임 결합
original_data_size = len(df_with_smoothed_features_train) # 원본 데이터의 전체 크기
combined_df = pd.concat([df_with_smoothed_features_train, synthetic_fraud_data_final], ignore_index=True)

print(f"CTGAN 후 결합된 DataFrame 크기: {original_data_size} (전처리된 원본) + {augmented_data_size} (합성 사기) = {len(combined_df)} (총계)")
print("-" * 50)

# 결합된 데이터프레임의 is_fraud 분포를 확인합니다.
print("\n결합된 데이터프레임의 'is_fraud' 분포:")
print(combined_df['is_fraud'].value_counts())
if 1 in combined_df['is_fraud'].value_counts() and 0 in combined_df['is_fraud'].value_counts():
    print(f"불균형 비율 (0:1): {combined_df['is_fraud'].value_counts()[0] / combined_df['is_fraud'].value_counts()[1]:.2f}:1")
else:
    print("불균형 비율 계산 불가 (0 또는 1 클래스가 없음).")

combined_df = combined_df.drop('cc_num_avg_amt', axis=1)

합성 데이터에 'cc_num' 재할당 중 (유사성 기반 매칭)...
CTGAN 후 결합된 DataFrame 크기: 1296675 (전처리된 원본) + 55000 (합성 사기) = 1351675 (총계)
--------------------------------------------------

결합된 데이터프레임의 'is_fraud' 분포:
is_fraud
0    1289169
1      62506
Name: count, dtype: int64
불균형 비율 (0:1): 20.62:1


In [ ]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1351675 entries, 0 to 1351674
Data columns (total 31 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   cc_num                   1351675 non-null  int64  
 1   job                      1351675 non-null  float64
 2   is_fraud                 1351675 non-null  int64  
 3   amt_log_std              1351675 non-null  float64
 4   day_of_week              1351675 non-null  int32  
 5   month                    1351675 non-null  int32  
 6   trans_hour_sin           1351675 non-null  float64
 7   trans_hour_cos           1351675 non-null  float64
 8   is_night                 1351675 non-null  int32  
 9   is_high_risk_period      1351675 non-null  int32  
 10  cnt_1d                   1351675 non-null  int64  
 11  cnt_7d                   1351675 non-null  int64  
 12  cnt_30d                  1351675 non-null  int64  
 13  time_since_last_trans    1351675 non-null 

In [ ]:
combined_df.shape

(1351675, 31)

In [ ]:
# --- 5. preprocess_data 선택 전처리 실행 ---
print("--- 5. preprocess_data 선택 전처리 실행 ---")
final_processed_df_train = preprocess_data(combined_df, hour='sincos', high_risk_period=True, age_group=True)
final_processed_df_test = preprocess_data(df_with_smoothed_features_test, hour='sincos', high_risk_period=True, age_group=True)

print(f"final_processed_df_train > preprocess_data 적용 후 DataFrame 형태: {final_processed_df_train.shape}")
print("final_processed_df_train > base_df 및 preprocess_data 후 컬럼 목록:", final_processed_df_train.columns.tolist())
print(f"final_processed_df_test > preprocess_data 적용 후 DataFrame 형태: {final_processed_df_test.shape}")
print("final_processed_df_test > base_df 및 preprocess_data 후 컬럼 목록:", final_processed_df_test.columns.tolist())
print("-" * 50)

--- 5. preprocess_data 선택 전처리 실행 ---
final_processed_df_train > preprocess_data 적용 후 DataFrame 형태: (1351675, 29)
final_processed_df_train > base_df 및 preprocess_data 후 컬럼 목록: ['cc_num', 'job', 'is_fraud', 'amt_log_std', 'day_of_week', 'month', 'trans_hour_sin', 'trans_hour_cos', 'is_high_risk_period', 'cnt_1d', 'cnt_7d', 'cnt_30d', 'time_since_last_trans', 'age_group', 'city_state', 'category_food_dining', 'category_gas_transport', 'category_grocery_net', 'category_grocery_pos', 'category_health_fitness', 'category_home', 'category_kids_pets', 'category_misc_net', 'category_misc_pos', 'category_personal_care', 'category_shopping_net', 'category_shopping_pos', 'category_travel', 'gender_M']
final_processed_df_test > preprocess_data 적용 후 DataFrame 형태: (555719, 29)
final_processed_df_test > base_df 및 preprocess_data 후 컬럼 목록: ['cc_num', 'job', 'is_fraud', 'amt_log_std', 'day_of_week', 'month', 'trans_hour_sin', 'trans_hour_cos', 'is_high_risk_period', 'cnt_1d', 'cnt_7d', 'cnt_30d', 'time_s

In [ ]:
# # --- 6. DGL 그래프 구축 및 GNN-CL 모델 입력 데이터 준비 ---
# print("--- 6. DGL 이종 그래프 구축 및 GNN-CL 모델 입력 준비 ---")

# # 학습 그래프 구축
# g_train, labels_for_gnn_train = build_heterogeneous_graph(final_processed_df_train)
# print(f"생성된 DGL 학습 그래프 객체: {g_train}")
# print(f"학습 그래프 트랜잭션 노드 (feat)의 원본 특징 형태: {g_train.nodes['transaction'].data['feat'].shape}")
# print(f"학습 그래프 타겟 레이블의 형태: {labels_for_gnn_train.shape}")

# # 테스트 그래프 구축
# g_test, labels_for_gnn_test = build_heterogeneous_graph(final_processed_df_test)
# print(f"생성된 DGL 테스트 그래프 객체: {g_test}")
# print(f"테스트 그래프 트랜잭션 노드 (feat)의 원본 특징 형태: {g_test.nodes['transaction'].data['feat'].shape}")
# print(f"테스트 그래프 타겟 레이블의 형태: {labels_for_gnn_test.shape}")
# print("-" * 50)

In [ ]:
global_id_maps = {
    'cc_num_to_idx': {},
    'category_to_idx': {},
    'city_state_to_idx': {}
}
global_num_unique_nodes = {
    'credit_card': 0,
    'original_category': 0,
    'city_state_node': 0
}


print("--- DGL 그래프 객체 생성 시작 ---")

# 훈련 그래프 생성 (is_train=True)
# 이 과정에서 global_id_maps와 global_num_unique_nodes가 채워집니다.
g_train, labels_for_gnn_train = build_heterogeneous_graph(final_processed_df_train, is_train=True, id_maps=global_id_maps)
print(f"생성된 DGL 학습 그래프 객체: {g_train}")
print(f"학습 그래프 트랜잭션 노드 (feat)의 원본 특징 형태: {g_train.nodes['transaction'].data['feat'].shape}")
print(f"학습 그래프 타겟 레이블의 형태: {labels_for_gnn_train.shape}")

# 테스트 그래프 생성 (is_train=False)
# 훈련 시 생성된 global_id_maps를 사용하여 노드 ID를 매핑합니다.
g_test, labels_for_gnn_test = build_heterogeneous_graph(final_processed_df_test, is_train=False, id_maps=global_id_maps)
print(f"생성된 DGL 테스트 그래프 객체: {g_test}")
print(f"테스트 그래프 트랜잭션 노드 (feat)의 원본 특징 형태: {g_test.nodes['transaction'].data['feat'].shape}")
print(f"테스트 그래프 타겟 레이블의 형태: {labels_for_gnn_test.shape}")

print("--- DGL 그래프 객체 생성 완료 ---")

--- DGL 그래프 객체 생성 시작 ---
생성된 DGL 학습 그래프 객체: Graph(num_nodes={'city_state_node': 57727, 'credit_card': 983, 'original_category': 14, 'transaction': 1351675},
      num_edges={('city_state_node', 'contains_trans_cs', 'transaction'): 1351675, ('credit_card', 'used_in', 'transaction'): 1351675, ('original_category', 'contains_trans_cat', 'transaction'): 1351675, ('transaction', 'in_category', 'original_category'): 1351675, ('transaction', 'in_city_state', 'city_state_node'): 1351675, ('transaction', 'uses_card', 'credit_card'): 1351675},
      metagraph=[('city_state_node', 'transaction', 'contains_trans_cs'), ('transaction', 'original_category', 'in_category'), ('transaction', 'city_state_node', 'in_city_state'), ('transaction', 'credit_card', 'uses_card'), ('credit_card', 'transaction', 'used_in'), ('original_category', 'transaction', 'contains_trans_cat')])
학습 그래프 트랜잭션 노드 (feat)의 원본 특징 형태: torch.Size([1351675, 27])
학습 그래프 타겟 레이블의 형태: torch.Size([1351675])
생성된 DGL 테스트 그래프 객체: Graph(num_n

In [ ]:
# --- 7. GNN-CL 모델 학습 루프 시작 ---
print("--- 7. GNN-CL 학습 루프 시작 ---")

device = torch.device('cpu')
# torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"현재 사용 중인 디바이스: {device}")

# 모델 초기화를 위한 노드 특징 차원 정보 준비
# GNN_CL_Model에 전달할 각 노드 타입의 원본 특징 차원
node_raw_dims = {
    'transaction': g_train.nodes['transaction'].data['feat'].shape[1],
    'credit_card': g_train.nodes['credit_card'].data['feat'].shape[1],
    'original_category': global_num_unique_nodes['original_category'], # 이 값은 훈련 시의 고유 카테고리 수 (14)
    'city_state_node': global_num_unique_nodes['city_state_node'] # 이 값은 훈련 시의 고유 city_state 수 (57746)
}

common_node_feature_dim = 27 # GNN_CL 모델이 기대하는 공통 특징 차원 (사전에 정의)
gnn_out_features = 64
cnn_out_channels = 32
cnn_kernel_size = 3
lstm_hidden_size = 16

learning_rate = 0.01
epochs = 30

# GNN_CL_Model 초기화: node_info 딕셔너리와 common_node_feature_dim을 전달합니다.
model = GNN_CL_Model(node_info=node_raw_dims,
                     common_node_feature_dim=common_node_feature_dim,
                     gnn_out_features=gnn_out_features,
                     cnn_out_channels=cnn_out_channels,
                     cnn_kernel_size=cnn_kernel_size,
                     lstm_hidden_size=lstm_hidden_size)

model.to(device)

# 옵티마이저 정의
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# --- 손실 함수 정의 (불균형 데이터셋을 위한 BCEWithLogitsLoss와 pos_weight) ---
# 'is_fraud' 분포에서 얻은 실제 값 사용 (CTGAN 후의 결합된 df 기준)
# 이 값들은 데이터 전처리 단계에서 얻은 실제 값으로 대체해야 합니다.
# 예시 값:
num_normal_combined = 1289169
num_fraud_combined = 62506

# pos_weight 계산: 정상 트랜잭션 수 / 사기 트랜잭션 수
pos_weight_value = num_normal_combined / num_fraud_combined
# 모델과 동일한 디바이스에 텐서로 변환하여 할당
pos_weight = torch.tensor([pos_weight_value], device=device, dtype=torch.float32)

# 손실 함수를 BCEWithLogitsLoss로 변경하고 pos_weight 적용
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"BCEWithLogitsLoss의 pos_weight 설정 값: {pos_weight_value:.2f}")
# --- 손실 함수 정의 끝 ---

# 그래프와 레이블을 디바이스로 이동 (이미 위에서 모델을 이동했으므로, 데이터도 이동해야 함)
labels_for_gnn = labels_for_gnn_train.to(device)
g = g_train.to(device) # 그래프 자체를 GPU로 옮김 (노드 특징도 함께 옮겨짐)

for epoch in range(epochs):
    model.train() # 모델을 훈련 모드로 설정
    optimizer.zero_grad() # 이전 기울기 초기화

    # 모델 forward: 이제 모델은 그래프 객체 'g'만 입력으로 받습니다.
    # 노드 특징은 모델 내부에서 g.nodes[ntype].data['feat']를 통해 직접 접근 및 변환됩니다.
    output = model(g)

    # --- 디버깅 출력 추가 ---
    print(f"Epoch {epoch+1} - Model output shape: {output.shape}, dtype: {output.dtype}")
    print(f"Epoch {epoch+1} - Labels shape: {labels_for_gnn.shape}, dtype: {labels_for_gnn.dtype}")
    # ---

    # 예측값(로짓)과 실제 레이블 간의 손실을 계산합니다.
    # output.squeeze()는 (N, 1) -> (N) 으로 만들어서 labels_for_gnn (N,)과 차원을 맞춥니다.
    loss = criterion(output.squeeze(), labels_for_gnn)

    loss.backward() # 역전파
    optimizer.step() # 가중치 업데이트

    if (epoch + 1) % 1 == 0:
        model.eval() # 평가 모드로 전환 (Dropout, BatchNorm 등 비활성화)
        with torch.no_grad(): # 기울기 계산 비활성화
            # BCEWithLogitsLoss를 사용했으므로, 확률로 변환하기 위해 sigmoid 적용
            probabilities = torch.sigmoid(output.squeeze())
            predicted = (probabilities > 0.5).float() # 임계값 0.5로 이진 예측

            accuracy = (predicted == labels_for_gnn).float().mean()
            print(f"에포크 {epoch+1}/{epochs}, 손실(Loss): {loss.item():.4f}, 정확도(Accuracy): {accuracy.item():.4f}")

print("--- GNN-CL 학습 완료 ---")

--- 7. GNN-CL 학습 루프 시작 ---
현재 사용 중인 디바이스: cpu
BCEWithLogitsLoss의 pos_weight 설정 값: 20.62
Epoch 1 - Model output shape: torch.Size([1351675, 1]), dtype: torch.float32
Epoch 1 - Labels shape: torch.Size([1351675]), dtype: torch.float32
에포크 1/30, 손실(Loss): 108.3104, 정확도(Accuracy): 0.0472
Epoch 2 - Model output shape: torch.Size([1351675, 1]), dtype: torch.float32
Epoch 2 - Labels shape: torch.Size([1351675]), dtype: torch.float32
에포크 2/30, 손실(Loss): 827.6334, 정확도(Accuracy): 0.9588
Epoch 3 - Model output shape: torch.Size([1351675, 1]), dtype: torch.float32
Epoch 3 - Labels shape: torch.Size([1351675]), dtype: torch.float32
에포크 3/30, 손실(Loss): 682.9309, 정확도(Accuracy): 0.9589
Epoch 4 - Model output shape: torch.Size([1351675, 1]), dtype: torch.float32
Epoch 4 - Labels shape: torch.Size([1351675]), dtype: torch.float32
에포크 4/30, 손실(Loss): 409.4031, 정확도(Accuracy): 0.9585
Epoch 5 - Model output shape: torch.Size([1351675, 1]), dtype: torch.float32
Epoch 5 - Labels shape: torch.Size([1351675]), 

In [ ]:
pip install koreanize_matplotlib -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, precision_score, recall_score, f1_score
import warnings
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns


warnings.filterwarnings('ignore')

# --- 8. 모델 검증 (Test Data) ---
print("\n--- 8. GNN-CL 모델 검증 (Test Data) ---")

# 테스트 데이터를 GPU로 이동
labels_for_gnn_test = labels_for_gnn_test.to(device)
g_test = g_test.to(device)

model.eval() # 모델을 평가 모드로 전환
with torch.no_grad(): # 역전파 계산 비활성화
    output_test = model(g_test)

    # 예측 확률
    probabilities = torch.sigmoid(output_test.squeeze()).cpu().numpy()

    # 이진 예측 (임계값 0.5)
    predictions = (probabilities > 0.5).astype(int)

    # 실제 레이블
    true_labels = labels_for_gnn_test.cpu().numpy()

# 1. Classification Report
print("\n--- Classification Report ---")
print(classification_report(true_labels, predictions, target_names=['Non-Fraud', 'Fraud']))

# 2. ROC AUC Score
try:
    roc_auc = roc_auc_score(true_labels, probabilities)
    print(f"\n--- ROC AUC Score: {roc_auc:.4f} ---")
except ValueError as e:
    print(f"\n--- ROC AUC Score: Calculation error: {e} (True labels might contain only one class) ---")

# 3. Precision, Recall, F1-Score 계산 및 출력
# 'pos_label=1'은 긍정 클래스(여기서는 'Fraud')에 대한 스코어를 계산하도록 지정합니다.
# 'average='binary''는 이진 분류에서 단일 클래스에 대한 스코어를 반환하도록 합니다.
# 'zero_division=0'은 분모가 0일 경우 0을 반환하도록 합니다.
precision = precision_score(true_labels, predictions, pos_label=1, average='binary', zero_division=0)
recall = recall_score(true_labels, predictions, pos_label=1, average='binary', zero_division=0)
f1 = f1_score(true_labels, predictions, pos_label=1, average='binary', zero_division=0)

print(f"\n--- 개별 성능 지표 (Fraud 클래스 기준) ---")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")


# 4. Confusion Matrix
cm = confusion_matrix(true_labels, predictions)
print(cm)


--- 8. GNN-CL 모델 검증 (Test Data) ---


: 